In [ ]:
# pip install -e ".[io]"  ← один раз в окружении
import os
import gc
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output

from brain_morph.utils import Volume, MeshTransformer3D
from brain_morph.utils import log_filter, histogram_matching, preprocess
from brain_morph.registration import (
    registration_cost,
    Stage, RegistrationPipeline,
    SAOptimizer, GradientOptimizer,
)

_cwd = Path(os.getcwd())
ROOT = str(_cwd.parent if _cwd.name == "notebooks" else _cwd)

print(f"torch {torch.__version__}  |  device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"ROOT: {ROOT}")

In [ ]:
BSCL_DIR = os.path.join(ROOT, 'data', 'BSCL')

MOVING_MASK = os.path.join(BSCL_DIR, 'C5-HC-4w', 'Pattern 1', 'Plane*.tif')
FIXED_MASK  = os.path.join(BSCL_DIR, 'C6_Tr_4w',  'Pattern 1', 'Plane*.tif')

OUTPUT_DIR  = os.path.join(ROOT, 'data', 'BSCL', 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SCALE_COARSE = 8    # XY downsample for SA stages
SCALE_FULL   = 1    # full resolution for chunked final transform
RATIO        = 1.6  # XY/Z voxel size ratio

print(f'Moving: {MOVING_MASK}')
print(f'Fixed:  {FIXED_MASK}')

In [ ]:
print('Loading moving (C5-HC-4w) ...')
vol_moving = Volume.load_tiff_series(MOVING_MASK, scale=SCALE_COARSE, ratio=RATIO)
print(f'  moving: {tuple(vol_moving.shape)}')

print('Loading fixed (C6_Tr_4w) ...')
vol_fixed = Volume.load_tiff_series(FIXED_MASK, scale=SCALE_COARSE, ratio=RATIO)
print(f'  fixed:  {tuple(vol_fixed.shape)}')

def _norm(v):
    t = v.float()
    return (t / t.max().clamp_min(1e-6)).as_subclass(Volume)

vol_moving = _norm(vol_moving)
vol_fixed  = _norm(vol_fixed)

D = vol_moving.shape[1]
vol_fixed = vol_fixed[:, :D].as_subclass(Volume)
H, W = vol_moving.shape[2], vol_moving.shape[3]

print(f'\nShape after align: moving={tuple(vol_moving.shape)}  fixed={tuple(vol_fixed.shape)}')

## Исходные изображения

In [ ]:
print('=== Moving (raw) ===')
vol_moving.visualize(channel=0)
print('=== Fixed (raw) ===')
vol_fixed.visualize(channel=0)

## Overlay до регистрации (moving=зелёный, fixed=красный)

In [ ]:
vol_moving.overlay(vol_fixed, channel=0)

## Предобработка: LoG-фильтр + совмещение гистограмм

In [ ]:
# LoG-фильтр (аналог mlog в MATLAB): выделяет контуры структур
# mode='log'        — только LoG         (для молодых мозгов, 'young')
# mode='log_binary' — LoG + бинаризация  (для взрослых, 'adult')
# mode='raw'        — без обработки

PREPROCESS_MODE = 'log'   # ← меняй здесь

im_moving_pp = preprocess(vol_moving, mode=PREPROCESS_MODE, sigma=5.0, size=5)
im_fixed_pp  = preprocess(vol_fixed,  mode=PREPROCESS_MODE, sigma=5.0, size=5)

# Совместить гистограммы moving → fixed
im_moving_pp = histogram_matching(im_moving_pp, im_fixed_pp)

print(f'Preprocessed: moving={tuple(im_moving_pp.shape)}  fixed={tuple(im_fixed_pp.shape)}')
print(f'Moving range: [{im_moving_pp.min():.3f}, {im_moving_pp.max():.3f}]')

In [ ]:
print('=== Moving (preprocessed) ===')
Volume(im_moving_pp).visualize(channel=0)
print('=== Fixed (preprocessed) ===')
Volume(im_fixed_pp).visualize(channel=0)

## Авто-маска ткани

In [ ]:
# Otsu-порог + morphological closing: маска подавляет регуляризацию
# в пустых областях (оторванная ОЛ, воздух)
mask_moving = vol_moving.auto_mask(closing_radius=5)  # (1, D, H, W) bool

print(f'Mask shape: {tuple(mask_moving.shape)}')
print(f'Tissue voxels: {mask_moving.sum().item():,} / {mask_moving.numel():,}  '
      f'({100*mask_moving.float().mean():.1f}%)')

Volume(mask_moving.float()).visualize(channel=0)

## Pipeline регистрации

In [ ]:
cost_log = []

def _cb(step, cost, warped):
    cost_log.append((step, cost))

_sa_kw = dict(
    coeff_start=0.12, coeff_drop=0.9966,
    attention_freq=50,
    callback=_cb, callback_freq=50,
    similarity='ncc',
)

sa_1 = SAOptimizer(temp_start=2e-3, temp_end=2e-5, **_sa_kw)
sa_2 = SAOptimizer(temp_start=1e-3, temp_end=1e-5, **_sa_kw)
sa_3 = SAOptimizer(temp_start=5e-4, temp_end=5e-6, **_sa_kw)
gd   = GradientOptimizer(
    optimizer_cls=torch.optim.Adam, lr=5e-4,
    tension_mode='squared', similarity='ncc',
)

# Регистрация ведётся на preprocessed изображениях + с маской
pipeline = RegistrationPipeline([
    Stage(grid_shape=(2, 2, 2), optimizer=sa_1, n_steps=2000, lam=5e-2, image_scale=8),
    Stage(grid_shape=(4, 4, 3), optimizer=sa_2, n_steps=2000, lam=2e-2, image_scale=8),
    Stage(grid_shape=(5, 5, 4), optimizer=sa_3, n_steps=2000, lam=1e-2, image_scale=4),
    Stage(grid_shape=(5, 5, 4), optimizer=gd,   n_steps=400,  lam=2e-3, image_scale=1),
])
print('Pipeline ready.')

In [ ]:
cost_log.clear()

# Регистрируем preprocessed; маска — для regularization
grid_result = pipeline.run(
    Volume(im_moving_pp),
    Volume(im_fixed_pp),
    mask=mask_moving[0],   # (D, H, W) bool
)
clear_output(wait=True)
print(f'Done. Grid: {grid_result.shape}')

## Результат на уменьшенном изображении

In [ ]:
grid_init_final = torch.stack(
    torch.meshgrid(*[torch.linspace(-1, 1, s) for s in grid_result.shape[:3]], indexing='ij'),
    dim=-1,
)
t_final = MeshTransformer3D(grid_init_final, (D, H, W))

with torch.no_grad():
    warped_coarse = t_final.transform(vol_moving, grid_result)

cost_id  = registration_cost(vol_moving, vol_fixed, t_final, grid_init_final, lam=0, similarity='ncc').item()
cost_res = registration_cost(vol_moving, vol_fixed, t_final, grid_result,      lam=0, similarity='ncc').item()

print(f'NCC identity: {cost_id:.4f}')
print(f'NCC result:   {cost_res:.4f}  (Δ = {cost_res - cost_id:+.4f})')

## Overlay после регистрации

In [ ]:
print('=== После регистрации ===')
Volume(warped_coarse).overlay(vol_fixed, channel=0)

diff_before = (vol_moving   - vol_fixed).abs()
diff_after  = (warped_coarse - vol_fixed).abs()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = max(diff_before.amax().item(), diff_after.amax().item())
axes[0].imshow(diff_before[0].amax(0).cpu(), cmap='hot', vmin=0, vmax=vmax)
axes[0].set_title(f'|diff| before  NCC={cost_id:.3f}'); axes[0].axis('off')
axes[1].imshow(diff_after[0].amax(0).cpu(),  cmap='hot', vmin=0, vmax=vmax)
axes[1].set_title(f'|diff| after   NCC={cost_res:.3f}'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## График метрики по стадиям

In [ ]:
if cost_log:
    import numpy as np
    stages_data, current = [], []
    for i, (s, c) in enumerate(cost_log):
        if i > 0 and s < cost_log[i-1][0]:
            stages_data.append(current); current = []
        current.append((s, c))
    stages_data.append(current)

    COLORS = ["#4C8BF5", "#E8555A", "#2ECC71", "#F5A623"]
    n = len(stages_data)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5))
    if n == 1: axes = [axes]
    for i, (stage_data, ax) in enumerate(zip(stages_data, axes)):
        steps_, costs_ = zip(*stage_data)
        color = COLORS[i % len(COLORS)]
        ax.plot(steps_, costs_, color=color, alpha=0.3, linewidth=0.7)
        w = max(1, len(costs_) // 20)
        sm = np.convolve(costs_, np.ones(w)/w, mode='same')
        ax.plot(steps_, sm, color=color, linewidth=2)
        ax.set_title(f'Stage {i+1}', fontsize=10)
        ax.set_xlabel('Step', fontsize=9)
        if i == 0: ax.set_ylabel('NCC', fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=8)
    plt.suptitle('Registration cost per stage', fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()

## Применение деформации к полному разрешению (chunked)

In [ ]:
print('Загрузка full-resolution moving ...')
vol_moving_full = Volume.load_tiff_series(MOVING_MASK, scale=SCALE_FULL, ratio=RATIO)
vol_moving_full = _norm(vol_moving_full)
D_f, H_f, W_f = vol_moving_full.shape[1], vol_moving_full.shape[2], vol_moving_full.shape[3]
print(f'Full resolution: {tuple(vol_moving_full.shape)}')

In [ ]:
# MeshTransformer3D не нужен — transform_chunked сам интерполирует сетку
# Используем t_final как носитель grid_init (только для создания объекта)
grid_init_full = torch.stack(
    torch.meshgrid(*[torch.linspace(-1, 1, s) for s in grid_result.shape[:3]], indexing='ij'),
    dim=-1,
)
t_full = MeshTransformer3D(grid_init_full, (D_f, H_f, W_f))

print(f'Applying deformation in chunks (chunk_size=50) ...')
warped_full = t_full.transform_chunked(vol_moving_full, grid_result, chunk_size=50)
gc.collect()
print(f'Done. Warped shape: {tuple(warped_full.shape)}')

In [ ]:
print('=== Warped full-resolution ===')
Volume(warped_full).visualize(channel=0)

## Сохранение результатов

In [ ]:
import numpy as np

# Warped full-resolution → NIfTI
out_nii  = os.path.join(OUTPUT_DIR, 'C5_warped_to_C6_full.nii.gz')
out_grid = os.path.join(OUTPUT_DIR, 'C5_to_C6_grid.pt')

arr = warped_full.cpu().numpy().astype(np.float32)
import nibabel as nib
nib.save(nib.Nifti1Image(arr[0], affine=np.eye(4)), out_nii)
torch.save(grid_result, out_grid)

print(f'Saved:\n  {out_nii}\n  {out_grid}')

In [ ]:
# Загрузка сохранённого результата:
# grid_saved = torch.load(out_grid)
# t_saved = MeshTransformer3D(
#     torch.stack(torch.meshgrid(*[torch.linspace(-1,1,s) for s in grid_saved.shape[:3]], indexing='ij'), dim=-1),
#     (D_f, H_f, W_f),
# )
# warped_new = t_saved.transform_chunked(vol_moving_full, grid_saved, chunk_size=50)